# Career Recommendation System — High-Accuracy Training Pipeline

**All outputs are shown in the Colab cell AND saved to Google Drive.**

### Folder structure saved to Drive
```
CareerRecommendation_Clean/
├── data/                  ← your CSV dataset
├── reports/
│   ├── text/              ← .txt classification reports & logs
│   ├── csv/               ← every table as CSV
│   └── run_metadata.json
├── figures/
│   ├── eda/               ← distributions, class balance, correlation
│   ├── model/             ← confusion matrix, per-class recall, CV bar
│   └── shap/              ← SHAP bar chart + summary plot
└── models/                ← all .pkl files (backend-ready)
```

Run cells **top to bottom** in a fresh Colab runtime (Runtime → Run all).


## CELL 1 — Install packages

In [ ]:
!pip -q install -U xgboost lightgbm catboost shap scikit-learn seaborn \
    joblib optuna imbalanced-learn
print("All packages ready.")

## CELL 2 — Mount Google Drive and create output folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, json, time, warnings, sys, pickle, joblib
from datetime import datetime

# ── Drive folders ─────────────────────────────────────────────────────────────
PROJECT_DIR   = Path('/content/drive/MyDrive/CareerRecommendation_Clean')
DATA_DIR      = PROJECT_DIR / 'data'
REPORT_DIR    = PROJECT_DIR / 'reports'
REPORT_TXT    = REPORT_DIR  / 'text'
REPORT_CSV    = REPORT_DIR  / 'csv'
FIG_DIR       = PROJECT_DIR / 'figures'
FIG_EDA       = FIG_DIR     / 'eda'
FIG_MODEL     = FIG_DIR     / 'model'
FIG_SHAP      = FIG_DIR     / 'shap'
MODEL_DIR     = PROJECT_DIR / 'models'

# ── Local folders (faster I/O during training) ────────────────────────────────
LOCAL         = Path('/content/career_work')
L_REPORT_TXT  = LOCAL / 'reports' / 'text'
L_REPORT_CSV  = LOCAL / 'reports' / 'csv'
L_FIG_EDA     = LOCAL / 'figures' / 'eda'
L_FIG_MODEL   = LOCAL / 'figures' / 'model'
L_FIG_SHAP    = LOCAL / 'figures' / 'shap'
L_MODEL       = LOCAL / 'models'

for p in [DATA_DIR, REPORT_DIR, REPORT_TXT, REPORT_CSV,
          FIG_DIR, FIG_EDA, FIG_MODEL, FIG_SHAP, MODEL_DIR,
          L_REPORT_TXT, L_REPORT_CSV,
          L_FIG_EDA, L_FIG_MODEL, L_FIG_SHAP, L_MODEL]:
    p.mkdir(parents=True, exist_ok=True)

print("Drive project root:", PROJECT_DIR)
print("Local  work  root :", LOCAL)
print()
print("Folder structure created:")
for p in [DATA_DIR, REPORT_TXT, REPORT_CSV,
          FIG_EDA, FIG_MODEL, FIG_SHAP, MODEL_DIR]:
    print(" ", p)

## CELL 3 — Upload or locate the dataset

In [ ]:
from google.colab import files

# ── Option A: already in Drive — uncomment and set path ──────────────────────
# DATA_PATH = PROJECT_DIR / 'data' / 'career_dataset.csv'

# ── Option B: manual upload ───────────────────────────────────────────────────
DATA_PATH = None

if DATA_PATH is None or not Path(str(DATA_PATH)).exists():
    print("Upload career_dataset.csv:")
    uploaded    = files.upload()
    fname       = next(iter(uploaded))
    DATA_PATH   = Path(f'/content/{fname}')
    shutil.copy2(DATA_PATH, DATA_DIR / 'career_dataset.csv')
    print("Saved copy to Drive:", DATA_DIR / 'career_dataset.csv')

DATA_PATH = Path(str(DATA_PATH))
assert DATA_PATH.exists(), f"Not found: {DATA_PATH}"
print("Dataset path:", DATA_PATH)

## CELL 4 — Imports, configuration and logging setup

In [ ]:
import io, warnings, contextlib
import numpy  as np
import pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, shap, xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

from scipy.optimize import minimize
from sklearn.compose        import ColumnTransformer
from sklearn.pipeline       import Pipeline
from sklearn.impute         import SimpleImputer
from sklearn.preprocessing  import OneHotEncoder, LabelEncoder, StandardScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_validate)
from sklearn.ensemble        import RandomForestClassifier
from sklearn.metrics         import (accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix,
    top_k_accuracy_score)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.feature_selection  import mutual_info_classif
from imblearn.over_sampling     import SMOTE

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION  ← edit only this block
# ══════════════════════════════════════════════════════════════════════════════
TARGET                = 'Career_Label'
RANDOM_STATE          = 42
TEST_SIZE             = 0.20
VAL_FRACTION          = 0.20
REMOVE_SPECIALIZATION = True      # drop Specialization to prevent leakage
USE_GPU               = True      # uses Colab T4/A100 GPU
RUN_OPTUNA            = False     # True → ~30 min Optuna HPO on XGBoost
OPTUNA_TRIALS         = 40
USE_SMOTE             = False     # True → SMOTE minority oversampling
# ══════════════════════════════════════════════════════════════════════════════

# ── Logger: mirrors every print to a .txt file ────────────────────────────────
LOG_PATH     = L_REPORT_TXT / 'training_log.txt'
_orig_stdout = sys.stdout

class TeeStream:
    def __init__(self, console, path):
        self.console = console
        self.file    = open(path, 'w', encoding='utf-8')
    def write(self, msg):
        try:  self.console.write(msg)
        except Exception: pass
        try:  self.file.write(msg)
        except Exception: pass
    def flush(self):
        try:  self.console.flush()
        except Exception: pass
        try:  self.file.flush()
        except Exception: pass
    def isatty(self): return False
    def close(self):
        try: self.file.close()
        except Exception: pass

sys.stdout = TeeStream(_orig_stdout, LOG_PATH)

# ── Figure helper: shows in cell AND saves to local + Drive ───────────────────
def save_fig(filename, local_folder, drive_folder, dpi=160):
    local_folder  = Path(local_folder);  local_folder.mkdir(parents=True, exist_ok=True)
    drive_folder  = Path(drive_folder);  drive_folder.mkdir(parents=True, exist_ok=True)
    local_path    = local_folder  / filename
    drive_path    = drive_folder  / filename
    plt.tight_layout()
    plt.savefig(local_path,  dpi=dpi, bbox_inches='tight')
    plt.savefig(drive_path,  dpi=dpi, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f"  Saved figure: {filename} -> {drive_folder}")

# ── CSV helper ────────────────────────────────────────────────────────────────
def save_csv(df, filename):
    lp = L_REPORT_CSV / filename
    dp = REPORT_CSV   / filename
    df.to_csv(lp);  df.to_csv(dp)
    print(f"  Saved CSV   : {filename} -> {REPORT_CSV}")

# ── TXT helper ────────────────────────────────────────────────────────────────
def save_txt(text, filename):
    lp = L_REPORT_TXT / filename
    dp = REPORT_TXT   / filename
    lp.write_text(text, encoding='utf-8')
    dp.write_text(text, encoding='utf-8')
    print(f"  Saved TXT   : {filename} -> {REPORT_TXT}")

plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid')

print('='*80)
print('CAREER RECOMMENDATION — HIGH-ACCURACY TRAINING PIPELINE')
print('='*80)
print('Started:', datetime.now().isoformat())
print('Config:')
print(f'  TARGET={TARGET}  RANDOM_STATE={RANDOM_STATE}')
print(f'  TEST_SIZE={TEST_SIZE}  VAL_FRACTION={VAL_FRACTION}')
print(f'  REMOVE_SPECIALIZATION={REMOVE_SPECIALIZATION}')
print(f'  USE_GPU={USE_GPU}  RUN_OPTUNA={RUN_OPTUNA}  USE_SMOTE={USE_SMOTE}')

## CELL 5 — Load dataset and basic inspection

In [ ]:
df_raw = pd.read_csv(DATA_PATH)

print('='*80)
print('DATASET OVERVIEW')
print('='*80)
print(f'Shape          : {df_raw.shape}')
print(f'Rows           : {len(df_raw):,}')
print(f'Columns        : {df_raw.shape[1]}')
print(f'Duplicate rows : {df_raw.duplicated().sum():,}')
print(f'Missing total  : {df_raw.isna().sum().sum():,}')
print()

if TARGET not in df_raw.columns:
    raise ValueError(
        f"Column '{TARGET}' not found!\n"
        f"Available: {df_raw.columns.tolist()}"
    )

# Show first rows
print('First 5 rows:')
display(df_raw.head())
print()
print('Data types:')
display(df_raw.dtypes.to_frame('dtype'))

# Save describe CSV
desc = df_raw.describe(include='all').T
save_csv(desc, '00_raw_describe.csv')

## CELL 6 — EDA: missing values and duplicates

In [ ]:
print('='*80)
print('MISSING VALUES')
print('='*80)

missing     = df_raw.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df  = pd.DataFrame({'count': missing, 'pct': missing_pct})
has_missing = missing_df[missing_df['count'] > 0]

if has_missing.empty:
    print('No missing values found.')
else:
    print('Columns with missing values:')
    display(has_missing)

save_csv(missing_df, '01_missing_values.csv')

# Duplicate report
dups = df_raw.duplicated().sum()
print(f'\nDuplicate rows: {dups}')

# Logical range checks
print('\nLOGICAL RANGE CHECKS')
print('-'*40)
range_rules = {
    'CGPA':                  (0, 10),
    'Attendance_Percentage': (0, 100),
    'Semester_Marks_Percent':(0, 100),
    'Internal_Marks':        (0, 100),
    'Practical_Marks':       (0, 100),
    'Project_Score':         (0, 100),
    'Lab_Score':             (0, 100),
    'Assignment_Score':      (0, 100),
    'Academic_Score':        (0, 100),
}
inv_rows = []
for col, (lo, hi) in range_rules.items():
    if col in df_raw.columns:
        mask = df_raw[col].notna() & ((df_raw[col] < lo) | (df_raw[col] > hi))
        cnt  = int(mask.sum())
        inv_rows.append({'column': col, 'min_allowed': lo,
                         'max_allowed': hi, 'invalid_count': cnt})
        if cnt:
            print(f'  INVALID: {col}: {cnt} values outside [{lo}, {hi}]')

inv_df = pd.DataFrame(inv_rows)
display(inv_df)
save_csv(inv_df, '02_invalid_value_report.csv')
print('Range check complete.')

## CELL 7 — EDA: career class distribution

In [ ]:
print('='*80)
print('CAREER CLASS DISTRIBUTION')
print('='*80)

career_counts = df_raw[TARGET].value_counts()
career_pct    = career_counts / len(df_raw) * 100
imbalance     = career_counts.max() / career_counts.min()

class_df = pd.DataFrame({'count': career_counts,
                         'percentage': career_pct.round(2)})
display(class_df)

print(f'\nNumber of classes : {len(career_counts)}')
print(f'Largest class     : {career_counts.max():,}')
print(f'Smallest class    : {career_counts.min():,}')
print(f'Imbalance ratio   : {imbalance:.2f}:1')
if imbalance > 3:
    print('  -> Noticeable imbalance — sample_weight will be applied.')
else:
    print('  -> Distribution is reasonably balanced.')

save_csv(class_df, '03_career_class_distribution.csv')

# Figure
plt.figure(figsize=(12, max(6, len(career_counts)*0.38)))
career_counts.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Career Class Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Number of Students')
plt.ylabel('Career')
for i, v in enumerate(career_counts.sort_values().values):
    plt.text(v+5, i, str(v), va='center', fontsize=8)
save_fig('01_career_class_distribution.png', L_FIG_EDA, FIG_EDA)

## CELL 8 — EDA: numerical distributions, skewness, outliers

In [ ]:
print('='*80)
print('NUMERICAL FEATURE ANALYSIS')
print('='*80)

num_cols = df_raw.select_dtypes(include=np.number).columns.tolist()
if TARGET in num_cols: num_cols.remove(TARGET)
print(f'Numeric columns: {len(num_cols)}')

# Stats table
desc = df_raw[num_cols].describe().T
desc['skewness'] = df_raw[num_cols].skew()
desc['kurtosis'] = df_raw[num_cols].kurt()
display(desc)
save_csv(desc, '04_numeric_statistics.csv')

# Outlier report
outlier_rows = []
for col in num_cols:
    s = df_raw[col].dropna()
    q1, q3  = s.quantile(0.25), s.quantile(0.75)
    iqr     = q3 - q1
    lo, hi  = q1 - 1.5*iqr, q3 + 1.5*iqr
    cnt     = int(((s < lo) | (s > hi)).sum())
    outlier_rows.append({'feature': col, 'Q1': round(q1,2), 'Q3': round(q3,2),
                         'IQR': round(iqr,2), 'lower': round(lo,2),
                         'upper': round(hi,2), 'outlier_count': cnt,
                         'outlier_pct': round(cnt/len(s)*100,2)})
out_df = pd.DataFrame(outlier_rows).sort_values('outlier_count', ascending=False)
print('\nOutlier report (IQR method):')
display(out_df)
save_csv(out_df, '05_outlier_report.csv')

# Histograms grid
n_cols_plot = 4
n_rows_plot = -(-len(num_cols) // n_cols_plot)
fig, axes = plt.subplots(n_rows_plot, n_cols_plot,
                         figsize=(18, 4*n_rows_plot))
axes = np.array(axes).flatten()
for ax, col in zip(axes, num_cols):
    ax.hist(df_raw[col].dropna(), bins=30, color='steelblue',
            edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=8)
    ax.tick_params(labelsize=7)
for ax in axes[len(num_cols):]: ax.axis('off')
plt.suptitle('Numerical Feature Distributions', fontsize=13,
             fontweight='bold', y=1.01)
save_fig('02_numerical_distributions.png', L_FIG_EDA, FIG_EDA)

## CELL 9 — EDA: categorical feature distributions

In [ ]:
print('='*80)
print('CATEGORICAL FEATURE ANALYSIS')
print('='*80)

cat_cols = df_raw.select_dtypes(
    include=['object','category','bool']).columns.tolist()
if TARGET in cat_cols: cat_cols.remove(TARGET)
print(f'Categorical columns: {len(cat_cols)}')

cat_summary = []
for col in cat_cols:
    cat_summary.append({
        'column':        col,
        'unique_values': df_raw[col].nunique(dropna=False),
        'missing':       int(df_raw[col].isna().sum()),
        'top_value':     df_raw[col].mode(dropna=True).iloc[0]
                         if not df_raw[col].dropna().empty else 'N/A'
    })
cat_df = pd.DataFrame(cat_summary)
display(cat_df)
save_csv(cat_df, '06_categorical_summary.csv')

# Bar chart for each categorical column
for col in cat_cols:
    counts = df_raw[col].fillna('MISSING').value_counts().head(20)
    plt.figure(figsize=(9, max(4, len(counts)*0.35)))
    counts.sort_values().plot(kind='barh', color='teal', edgecolor='white')
    plt.title(f'{col} — Top Categories', fontweight='bold')
    plt.xlabel('Count')
    safe = ''.join(c if c.isalnum() else '_' for c in col)
    save_fig(f'cat_{safe}.png', L_FIG_EDA, FIG_EDA)

## CELL 10 — EDA: correlation heatmap and strong pairs

In [ ]:
print('='*80)
print('CORRELATION ANALYSIS')
print('='*80)

corr = df_raw[num_cols].corr()
save_csv(corr, '07_correlation_matrix.csv')

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, cmap='coolwarm', center=0, mask=mask,
            annot=len(num_cols) <= 15, fmt='.2f',
            linewidths=0.4, vmin=-1, vmax=1)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
save_fig('03_correlation_heatmap.png', L_FIG_EDA, FIG_EDA)

# Strong pairs
pairs = []
for i, c1 in enumerate(corr.columns):
    for c2 in corr.columns[i+1:]:
        v = corr.loc[c1, c2]
        if abs(v) >= 0.80:
            pairs.append({'feature_1': c1, 'feature_2': c2,
                          'correlation': round(v, 4)})
if pairs:
    strong_df = pd.DataFrame(pairs).sort_values(
        'correlation', key=abs, ascending=False)
    print('Strong correlations (|r| >= 0.80):')
    display(strong_df)
    save_csv(strong_df, '08_strong_correlations.csv')
else:
    print('No feature pairs with |r| >= 0.80')

## CELL 11 — EDA: mutual information (feature vs target)

In [ ]:
print('='*80)
print('MUTUAL INFORMATION')
print('='*80)

mi_df_data = df_raw[num_cols].fillna(df_raw[num_cols].median(numeric_only=True))
sample_n   = min(10000, len(mi_df_data))
mi_sample  = mi_df_data.sample(sample_n, random_state=RANDOM_STATE)
y_sample   = LabelEncoder().fit_transform(df_raw.loc[mi_sample.index, TARGET])

mi_scores = mutual_info_classif(mi_sample, y_sample, random_state=RANDOM_STATE)
mi_report = pd.DataFrame({'feature': num_cols, 'mutual_info': mi_scores})    .sort_values('mutual_info', ascending=False)

display(mi_report.head(30))
save_csv(mi_report, '09_mutual_information.csv')

plt.figure(figsize=(10, max(6, len(mi_report.head(20))*0.4)))
top_mi = mi_report.head(20).sort_values('mutual_info')
sns.barplot(data=top_mi, x='mutual_info', y='feature', palette='Blues_r')
plt.title('Top 20 Features by Mutual Information (vs Career)', fontweight='bold')
plt.xlabel('Mutual Information Score')
save_fig('04_mutual_information.png', L_FIG_EDA, FIG_EDA)

## CELL 12 — Leakage review and build X / y

In [ ]:
print('='*80)
print('LEAKAGE REVIEW & FEATURE SELECTION')
print('='*80)

if 'Specialization' in df_raw.columns:
    print(f'Specialization — unique: {df_raw["Specialization"].nunique()}  '
          f'missing: {df_raw["Specialization"].isna().sum()}')

df = df_raw.copy()
df = df.drop_duplicates().reset_index(drop=True)
print(f'Rows after deduplication: {len(df):,}')

drop_cols = [TARGET]
if REMOVE_SPECIALIZATION and 'Specialization' in df.columns:
    drop_cols.append('Specialization')
    print('Specialization DROPPED (leakage prevention).')
else:
    print('Specialization KEPT as input feature.')

X     = df.drop(columns=drop_cols).copy()
y_raw = df[TARGET].copy()

print(f'\nX shape   : {X.shape}')
print(f'y classes : {y_raw.nunique()}')
print('Input columns:')
for i, c in enumerate(X.columns, 1):
    print(f'  {i:2d}. {c}')

## CELL 13 — Stratified Train / Validation / Test split

In [ ]:
print('='*80)
print('DATA SPLIT (stratified)')
print('='*80)

X_temp, X_test,  y_temp_raw, y_test_raw = train_test_split(
    X, y_raw, test_size=TEST_SIZE,
    random_state=RANDOM_STATE, stratify=y_raw
)
X_train, X_val, y_train_raw, y_val_raw = train_test_split(
    X_temp, y_temp_raw, test_size=VAL_FRACTION,
    random_state=RANDOM_STATE, stratify=y_temp_raw
)

total = len(X)
split_df = pd.DataFrame({
    'split':      ['Train', 'Validation', 'Test'],
    'rows':       [len(X_train), len(X_val), len(X_test)],
    'percentage': [f'{len(X_train)/total*100:.1f}%',
                   f'{len(X_val)/total*100:.1f}%',
                   f'{len(X_test)/total*100:.1f}%']
})
display(split_df)
save_csv(split_df, '10_data_split.csv')

# Check class distribution consistency
print('\nClass distribution across splits:')
for name, y_s in [('Train', y_train_raw), ('Val', y_val_raw), ('Test', y_test_raw)]:
    vc = y_s.value_counts(normalize=True).mul(100).round(1)
    print(f'  {name}: {len(y_s):,} rows  |  '
          f'min class%={vc.min():.1f}  max class%={vc.max():.1f}')

## CELL 14 — Encode target labels

In [ ]:
print('='*80)
print('TARGET ENCODING')
print('='*80)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_raw)
y_val   = label_encoder.transform(y_val_raw)
y_test  = label_encoder.transform(y_test_raw)

class_df = pd.DataFrame({
    'index':  range(len(label_encoder.classes_)),
    'career': label_encoder.classes_
})
display(class_df)
save_csv(class_df, '11_career_classes.csv')

print(f'\nTotal classes: {len(label_encoder.classes_)}')
joblib.dump(label_encoder, L_MODEL / 'label_encoder.pkl')
print('label_encoder.pkl saved.')

## CELL 15 — Preprocessing pipeline (fit ONLY on train)

In [ ]:
print('='*80)
print('PREPROCESSING PIPELINE')
print('='*80)

numeric_features     = X_train.select_dtypes(
    include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(
    include=['object', 'category', 'bool']).columns.tolist()

print(f'Numeric features    : {len(numeric_features)}')
print(f'Categorical features: {len(categorical_features)}')
print()
print('Numeric   :', numeric_features)
print('Categorical:', categorical_features)

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=True)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline,     numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

# CRITICAL: fit_transform only on train
X_train_proc = preprocessor.fit_transform(X_train)
X_val_proc   = preprocessor.transform(X_val)
X_test_proc  = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out().tolist()

print(f'\nProcessed shapes:')
print(f'  Train : {X_train_proc.shape}')
print(f'  Val   : {X_val_proc.shape}')
print(f'  Test  : {X_test_proc.shape}')
print(f'  Total transformed features: {len(feature_names)}')

joblib.dump(preprocessor, L_MODEL / 'preprocessor.pkl')
print('preprocessor.pkl saved.')

## CELL 16 — Sample weights / SMOTE (class imbalance)

In [ ]:
print('='*80)
print('CLASS IMBALANCE HANDLING')
print('='*80)

if USE_SMOTE:
    from scipy.sparse import issparse
    X_tr_d = X_train_proc.toarray() if issparse(X_train_proc) else X_train_proc
    smote  = SMOTE(random_state=RANDOM_STATE, k_neighbors=3)
    X_train_proc, y_train = smote.fit_resample(X_tr_d, y_train)
    sample_weights = np.ones(len(y_train))
    print(f'SMOTE applied. Training rows: {X_train_proc.shape[0]:,}')
else:
    sample_weights = compute_sample_weight('balanced', y=y_train)
    print('Using sample_weight (balanced).')
    sw_series = pd.Series(sample_weights).describe()
    display(sw_series.to_frame('sample_weight_stats'))

## CELL 17 — Optuna HPO for XGBoost (optional, ~30 min)

In [ ]:
from scipy.sparse import issparse as _iss

best_xgb_params = None

if RUN_OPTUNA:
    print('='*80)
    print('OPTUNA HYPERPARAMETER OPTIMISATION')
    print('='*80)

    def _optuna_obj(trial):
        p = dict(
            objective       = 'multi:softprob',
            num_class       = len(label_encoder.classes_),
            tree_method     = 'hist',
            device          = 'cuda' if USE_GPU else 'cpu',
            eval_metric     = 'mlogloss',
            n_estimators    = trial.suggest_int('n_estimators', 400, 1500),
            max_depth       = trial.suggest_int('max_depth', 5, 10),
            learning_rate   = trial.suggest_float('lr', 0.01, 0.15, log=True),
            subsample       = trial.suggest_float('subsample', 0.6, 1.0),
            colsample_bytree= trial.suggest_float('colsample', 0.5, 1.0),
            min_child_weight= trial.suggest_int('mcw', 1, 10),
            gamma           = trial.suggest_float('gamma', 0, 0.5),
            reg_alpha       = trial.suggest_float('alpha', 0, 2.0),
            reg_lambda      = trial.suggest_float('lambda', 0.5, 5.0),
            random_state    = RANDOM_STATE, n_jobs=-1,
        )
        skf  = StratifiedKFold(n_splits=3, shuffle=True,
                               random_state=RANDOM_STATE)
        X_cv = X_train_proc.toarray() if _iss(X_train_proc) else X_train_proc
        scores = []
        for tr_i, va_i in skf.split(X_cv, y_train):
            m = xgb.XGBClassifier(**p, early_stopping_rounds=30)
            m.fit(X_cv[tr_i], y_train[tr_i],
                  sample_weight=sample_weights[tr_i],
                  eval_set=[(X_cv[va_i], y_train[va_i])], verbose=False)
            scores.append(accuracy_score(y_train[va_i],
                                         m.predict(X_cv[va_i])))
        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(_optuna_obj, n_trials=OPTUNA_TRIALS,
                   show_progress_bar=True)
    best_xgb_params = study.best_params
    opt_df = pd.DataFrame([best_xgb_params])
    display(opt_df)
    save_csv(opt_df, '12_optuna_best_params.csv')
    print('Best Optuna value:', study.best_value)
else:
    print('Optuna skipped (RUN_OPTUNA=False). Using tuned defaults.')

## CELL 18 — Train XGBoost

In [ ]:
print('='*80)
print('TRAINING: XGBoost')
print('='*80)

if best_xgb_params:
    xgb_p = dict(objective='multi:softprob',
                 num_class=len(label_encoder.classes_),
                 eval_metric='mlogloss', random_state=RANDOM_STATE,
                 n_jobs=-1, early_stopping_rounds=50,
                 tree_method='hist',
                 device='cuda' if USE_GPU else 'cpu',
                 **best_xgb_params)
else:
    xgb_p = dict(objective='multi:softprob',
                 num_class=len(label_encoder.classes_),
                 n_estimators=1200, max_depth=8, learning_rate=0.05,
                 subsample=0.85, colsample_bytree=0.75,
                 min_child_weight=2, gamma=0.05,
                 reg_alpha=0.1, reg_lambda=1.5,
                 eval_metric='mlogloss', random_state=RANDOM_STATE,
                 n_jobs=-1, early_stopping_rounds=50,
                 tree_method='hist',
                 device='cuda' if USE_GPU else 'cpu')

xgb_model = xgb.XGBClassifier(**xgb_p)
t0 = time.time()
xgb_model.fit(X_train_proc, y_train,
              sample_weight=sample_weights,
              eval_set=[(X_val_proc, y_val)],
              verbose=100)
xgb_time = (time.time()-t0)/60
print(f'\nXGBoost time     : {xgb_time:.2f} min')
print(f'Best iteration   : {getattr(xgb_model,"best_iteration","N/A")}')
print(f'Best val score   : {getattr(xgb_model,"best_score","N/A")}')

xgb_val_acc = accuracy_score(y_val, xgb_model.predict(X_val_proc))
print(f'XGBoost val acc  : {xgb_val_acc*100:.2f}%')

joblib.dump(xgb_model, L_MODEL / 'career_model.pkl', compress=3)
print('career_model.pkl saved.')

## CELL 19 — Train LightGBM

In [ ]:
print('='*80)
print('TRAINING: LightGBM')
print('='*80)

lgb_model = lgb.LGBMClassifier(
    objective='multiclass', num_class=len(label_encoder.classes_),
    n_estimators=1200, max_depth=8, learning_rate=0.05,
    num_leaves=80, subsample=0.85, colsample_bytree=0.75,
    min_child_samples=10, reg_alpha=0.1, reg_lambda=1.5,
    random_state=RANDOM_STATE, n_jobs=-1,
    device_type='gpu' if USE_GPU else 'cpu', verbose=-1,
)
t0 = time.time()
lgb_model.fit(X_train_proc, y_train,
              sample_weight=sample_weights,
              eval_set=[(X_val_proc, y_val)],
              callbacks=[lgb.early_stopping(50, verbose=False),
                         lgb.log_evaluation(100)])
lgb_time = (time.time()-t0)/60
print(f'\nLightGBM time    : {lgb_time:.2f} min')
print(f'Best iteration   : {lgb_model.best_iteration_}')

lgb_val_acc = accuracy_score(y_val, lgb_model.predict(X_val_proc))
print(f'LightGBM val acc : {lgb_val_acc*100:.2f}%')

joblib.dump(lgb_model, L_MODEL / 'lgbm_model.pkl', compress=3)
print('lgbm_model.pkl saved.')

## CELL 20 — Train CatBoost

In [ ]:
from scipy.sparse import issparse as _iss

print('='*80)
print('TRAINING: CatBoost')
print('='*80)

X_tr_cb  = X_train_proc.toarray() if _iss(X_train_proc) else X_train_proc
X_val_cb = X_val_proc.toarray()   if _iss(X_val_proc)   else X_val_proc

cb_model = CatBoostClassifier(
    iterations=800, depth=8, learning_rate=0.05,
    l2_leaf_reg=3.0, bagging_temperature=1.0, random_strength=1.0,
    loss_function='MultiClass', eval_metric='Accuracy',
    random_seed=RANDOM_STATE,
    task_type='GPU' if USE_GPU else 'CPU',
    early_stopping_rounds=50, verbose=100,
)
t0 = time.time()
cb_model.fit(X_tr_cb, y_train, sample_weight=sample_weights,
             eval_set=(X_val_cb, y_val), use_best_model=True)
cb_time = (time.time()-t0)/60
print(f'\nCatBoost time    : {cb_time:.2f} min')

cb_val_acc = accuracy_score(y_val, cb_model.predict(X_val_cb))
print(f'CatBoost val acc : {cb_val_acc*100:.2f}%')

joblib.dump(cb_model, L_MODEL / 'catboost_model.pkl', compress=3)
print('catboost_model.pkl saved.')

## CELL 21 — Train Random Forest

In [ ]:
print('='*80)
print('TRAINING: Random Forest')
print('='*80)

rf_model = RandomForestClassifier(
    n_estimators=400, max_depth=None, max_features='sqrt',
    min_samples_leaf=2, class_weight='balanced_subsample',
    random_state=RANDOM_STATE, n_jobs=-1,
)
t0 = time.time()
rf_model.fit(X_train_proc, y_train, sample_weight=sample_weights)
rf_time = (time.time()-t0)/60
print(f'\nRandom Forest time    : {rf_time:.2f} min')

rf_val_acc = accuracy_score(y_val, rf_model.predict(X_val_proc))
print(f'Random Forest val acc : {rf_val_acc*100:.2f}%')

joblib.dump(rf_model, L_MODEL / 'rf_model.pkl', compress=3)
print('rf_model.pkl saved.')

## CELL 22 — Individual model validation summary

In [ ]:
from scipy.sparse import issparse as _iss

print('='*80)
print('INDIVIDUAL MODEL VALIDATION RESULTS')
print('='*80)

X_val_d = X_val_proc.toarray() if _iss(X_val_proc) else X_val_proc

p_xgb = xgb_model.predict_proba(X_val_proc)
p_lgb = lgb_model.predict_proba(X_val_proc)
p_cb  = cb_model.predict_proba(X_val_d)
p_rf  = rf_model.predict_proba(X_val_proc)

individual = []
for name, proba in [('XGBoost', p_xgb), ('LightGBM', p_lgb),
                    ('CatBoost', p_cb),  ('RandomForest', p_rf)]:
    preds = proba.argmax(1)
    individual.append({
        'model':      name,
        'val_acc':    round(accuracy_score(y_val, preds)*100, 2),
        'val_f1_mac': round(f1_score(y_val, preds, average='macro',
                                     zero_division=0)*100, 2),
    })

indiv_df = pd.DataFrame(individual)
display(indiv_df)
save_csv(indiv_df, '13_individual_val_results.csv')

# Bar chart
plt.figure(figsize=(8, 4))
x = np.arange(len(indiv_df))
w = 0.35
plt.bar(x - w/2, indiv_df['val_acc'],   width=w, label='Accuracy',  color='steelblue')
plt.bar(x + w/2, indiv_df['val_f1_mac'],width=w, label='Macro F1',  color='coral')
plt.xticks(x, indiv_df['model'])
plt.ylabel('Score (%)')
plt.title('Individual Model Validation Results', fontweight='bold')
plt.legend(); plt.ylim(0, 105)
for i, row in indiv_df.iterrows():
    plt.text(i-w/2, row['val_acc']+0.5,   f"{row['val_acc']:.1f}",   ha='center', fontsize=8)
    plt.text(i+w/2, row['val_f1_mac']+0.5, f"{row['val_f1_mac']:.1f}", ha='center', fontsize=8)
save_fig('05_individual_model_val.png', L_FIG_MODEL, FIG_MODEL)

## CELL 23 — Ensemble: optimise soft-vote weights on validation

In [ ]:
import pickle as pkl
from scipy.sparse import issparse as _iss

print('='*80)
print('ENSEMBLE WEIGHT OPTIMISATION')
print('='*80)

def neg_acc(w):
    w = np.maximum(w, 0);  w = w / w.sum()
    comb = w[0]*p_xgb + w[1]*p_lgb + w[2]*p_cb + w[3]*p_rf
    return -accuracy_score(y_val, comb.argmax(1))

best_res = None
for _ in range(30):
    w0  = np.random.dirichlet(np.ones(4))
    res = minimize(neg_acc, w0, method='Nelder-Mead',
                   options={'maxiter': 3000, 'xatol': 1e-7, 'fatol': 1e-7})
    if best_res is None or res.fun < best_res.fun:
        best_res = res

raw_w       = np.maximum(best_res.x, 0)
ens_weights = (raw_w / raw_w.sum()).tolist()

ens_val_proba = (ens_weights[0]*p_xgb + ens_weights[1]*p_lgb +
                 ens_weights[2]*p_cb  + ens_weights[3]*p_rf)
ens_val_acc   = accuracy_score(y_val, ens_val_proba.argmax(1))
ens_val_f1    = f1_score(y_val, ens_val_proba.argmax(1),
                         average='macro', zero_division=0)

wt_df = pd.DataFrame({
    'model':  ['XGBoost', 'LightGBM', 'CatBoost', 'RandomForest'],
    'weight': [round(w, 4) for w in ens_weights],
})
display(wt_df)
print(f'\nEnsemble val accuracy : {ens_val_acc*100:.2f}%')
print(f'Ensemble val Macro F1 : {ens_val_f1*100:.2f}%')

save_csv(wt_df, '14_ensemble_weights.csv')

# Pie chart of weights
plt.figure(figsize=(6, 6))
plt.pie(ens_weights, labels=wt_df['model'], autopct='%1.1f%%',
        colors=['steelblue','coral','seagreen','mediumpurple'],
        startangle=140)
plt.title('Ensemble Soft-Vote Weights (optimised on Validation)',
          fontweight='bold')
save_fig('06_ensemble_weights_pie.png', L_FIG_MODEL, FIG_MODEL)

# Save weights pkl
with open(L_MODEL / 'ensemble_weights.pkl', 'wb') as f:
    pkl.dump(ens_weights, f)
print('ensemble_weights.pkl saved.')

## CELL 24 — Final test evaluation (test set used ONLY here)

In [ ]:
from scipy.sparse import issparse as _iss

print('='*80)
print('FINAL TEST EVALUATION — ENSEMBLE')
print('='*80)

X_test_d = X_test_proc.toarray() if _iss(X_test_proc) else X_test_proc

t_xgb = xgb_model.predict_proba(X_test_proc)
t_lgb = lgb_model.predict_proba(X_test_proc)
t_cb  = cb_model.predict_proba(X_test_d)
t_rf  = rf_model.predict_proba(X_test_proc)

test_proba = (ens_weights[0]*t_xgb + ens_weights[1]*t_lgb +
              ens_weights[2]*t_cb  + ens_weights[3]*t_rf)
test_pred  = test_proba.argmax(axis=1)

test_accuracy        = accuracy_score(y_test, test_pred)
test_macro_precision = precision_score(y_test, test_pred,
    average='macro', zero_division=0)
test_macro_recall    = recall_score(y_test, test_pred,
    average='macro', zero_division=0)
test_macro_f1        = f1_score(y_test, test_pred,
    average='macro', zero_division=0)
test_weighted_f1     = f1_score(y_test, test_pred,
    average='weighted', zero_division=0)
top3_accuracy        = top_k_accuracy_score(
    y_test, test_proba, k=3,
    labels=np.arange(len(label_encoder.classes_)))
top5_accuracy        = top_k_accuracy_score(
    y_test, test_proba, k=5,
    labels=np.arange(len(label_encoder.classes_)))

metrics_df = pd.DataFrame([{
    'metric': 'Accuracy',       'ensemble': f'{test_accuracy*100:.2f}%'},
    {'metric': 'Macro Precision', 'ensemble': f'{test_macro_precision*100:.2f}%'},
    {'metric': 'Macro Recall',    'ensemble': f'{test_macro_recall*100:.2f}%'},
    {'metric': 'Macro F1',        'ensemble': f'{test_macro_f1*100:.2f}%'},
    {'metric': 'Weighted F1',     'ensemble': f'{test_weighted_f1*100:.2f}%'},
    {'metric': 'Top-3 Accuracy',  'ensemble': f'{top3_accuracy*100:.2f}%'},
    {'metric': 'Top-5 Accuracy',  'ensemble': f'{top5_accuracy*100:.2f}%'},
])
display(metrics_df)
save_csv(metrics_df, '15_final_test_metrics.csv')

# Individual model test accuracies
print('\nIndividual model test accuracies:')
indiv_test = []
for name, pr in [('XGBoost',t_xgb),('LightGBM',t_lgb),
                 ('CatBoost',t_cb), ('RandomForest',t_rf)]:
    acc = accuracy_score(y_test, pr.argmax(1))*100
    f1  = f1_score(y_test, pr.argmax(1), average='macro', zero_division=0)*100
    print(f'  {name:14s}  acc={acc:.2f}%  macro_f1={f1:.2f}%')
    indiv_test.append({'model': name, 'test_acc': round(acc,2),
                       'test_macro_f1': round(f1,2)})
indiv_test_df = pd.DataFrame(indiv_test)
save_csv(indiv_test_df, '16_individual_test_results.csv')

## CELL 25 — Full classification report (per class)

In [ ]:
print('='*80)
print('CLASSIFICATION REPORT — ENSEMBLE (Test Set)')
print('='*80)

full_report = classification_report(
    y_test, test_pred,
    target_names=label_encoder.classes_,
    zero_division=0
)
print(full_report)
save_txt(full_report, '17_classification_report.txt')

# Per-class DataFrame
rpt_dict  = classification_report(
    y_test, test_pred,
    target_names=label_encoder.classes_,
    output_dict=True, zero_division=0)
per_class = pd.DataFrame(rpt_dict).T.loc[
    label_encoder.classes_,
    ['precision', 'recall', 'f1-score', 'support']
]
display(per_class)
save_csv(per_class, '18_per_class_metrics.csv')

## CELL 26 — Confusion matrix (image + CSV)

In [ ]:
cm = confusion_matrix(y_test, test_pred)

# ── Heatmap ──────────────────────────────────────────────────────────────────
plt.figure(figsize=(max(14, len(label_encoder.classes_)*0.6),
                    max(12, len(label_encoder.classes_)*0.55)))
sns.heatmap(cm, cmap='Blues', annot=len(label_encoder.classes_) <= 20,
            fmt='d', linewidths=0.4,
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Ensemble Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Career', fontsize=11)
plt.ylabel('Actual Career',    fontsize=11)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
save_fig('07_confusion_matrix.png', L_FIG_MODEL, FIG_MODEL)

# ── CSV ───────────────────────────────────────────────────────────────────────
cm_df = pd.DataFrame(cm, index=label_encoder.classes_,
                         columns=label_encoder.classes_)
save_csv(cm_df, '19_confusion_matrix.csv')

# ── Per-class recall bar chart ────────────────────────────────────────────────
plt.figure(figsize=(12, max(8, len(label_encoder.classes_)*0.4)))
per_class['recall'].sort_values().plot(
    kind='barh', color='steelblue', edgecolor='white')
plt.axvline(x=0.8, color='red', linestyle='--', alpha=0.6, label='0.80 target')
plt.title('Per-Class Recall (Test Set) — Ensemble', fontweight='bold')
plt.xlabel('Recall')
plt.legend()
save_fig('08_per_class_recall.png', L_FIG_MODEL, FIG_MODEL)

# ── Per-class F1 bar chart ────────────────────────────────────────────────────
plt.figure(figsize=(12, max(8, len(label_encoder.classes_)*0.4)))
per_class['f1-score'].sort_values().plot(
    kind='barh', color='seagreen', edgecolor='white')
plt.axvline(x=0.8, color='red', linestyle='--', alpha=0.6, label='0.80 target')
plt.title('Per-Class F1 Score (Test Set) — Ensemble', fontweight='bold')
plt.xlabel('F1 Score')
plt.legend()
save_fig('09_per_class_f1.png', L_FIG_MODEL, FIG_MODEL)

## CELL 27 — Sample top-5 career recommendations

In [ ]:
print('='*80)
print('SAMPLE TOP-5 RECOMMENDATIONS')
print('='*80)

top5_idx   = np.argsort(test_proba, axis=1)[:, -5:][:, ::-1]
top5_probs = np.take_along_axis(test_proba, top5_idx, axis=1)
top5_names = label_encoder.inverse_transform(
    top5_idx.ravel()).reshape(top5_idx.shape)

rows = []
for i in range(min(15, len(test_pred))):
    row = {'student': i, 'actual': label_encoder.classes_[y_test[i]]}
    for rank in range(5):
        row[f'top{rank+1}_career'] = top5_names[i, rank]
        row[f'top{rank+1}_prob']   = round(float(top5_probs[i, rank])*100, 1)
    rows.append(row)

sample_df = pd.DataFrame(rows)
display(sample_df)
save_csv(sample_df, '20_sample_top5_recommendations.csv')

## CELL 28 — 5-fold stratified cross-validation (XGBoost)

In [ ]:
from scipy.sparse import issparse as _iss

print('='*80)
print('5-FOLD CROSS-VALIDATION')
print('='*80)

skf      = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
X_cv_arr = X_train_proc.toarray() if _iss(X_train_proc) else X_train_proc
cv_scores, fold_rows = [], []

n_est = getattr(xgb_model, 'best_iteration', 500) + 50

for fold, (tr_i, va_i) in enumerate(skf.split(X_cv_arr, y_train), 1):
    m = xgb.XGBClassifier(
        **{k:v for k,v in xgb_p.items()
           if k not in ('early_stopping_rounds', 'n_estimators')},
        n_estimators=n_est,
        early_stopping_rounds=None,
    )
    m.fit(X_cv_arr[tr_i], y_train[tr_i],
          sample_weight=sample_weights[tr_i], verbose=False)
    acc = accuracy_score(y_train[va_i], m.predict(X_cv_arr[va_i]))
    f1  = f1_score(y_train[va_i], m.predict(X_cv_arr[va_i]),
                   average='macro', zero_division=0)
    cv_scores.append(acc)
    fold_rows.append({'fold': fold, 'accuracy': round(acc*100,2),
                      'macro_f1': round(f1*100,2)})
    print(f'  Fold {fold}: acc={acc*100:.2f}%  f1={f1*100:.2f}%')

cv_df  = pd.DataFrame(fold_rows)
cv_mean= np.mean(cv_scores)
cv_std = np.std(cv_scores)
print(f'\nMean CV Accuracy : {cv_mean*100:.2f}% ± {cv_std*100:.2f}%')

display(cv_df)
save_csv(cv_df, '21_cross_validation.csv')

plt.figure(figsize=(7, 4))
colors = ['steelblue']*5
plt.bar(cv_df['fold'], cv_df['accuracy'], color=colors, edgecolor='white')
plt.axhline(cv_mean*100, linestyle='--', color='red',
            label=f'Mean={cv_mean*100:.2f}%')
plt.xlabel('Fold'); plt.ylabel('Accuracy (%)')
plt.title('5-Fold Stratified CV (XGBoost)', fontweight='bold')
plt.legend(); plt.ylim(max(0, cv_mean*100 - 10), 100)
for i, row in cv_df.iterrows():
    plt.text(i+1, row['accuracy']+0.2, f"{row['accuracy']:.1f}%",
             ha='center', fontsize=9)
save_fig('10_cross_validation.png', L_FIG_MODEL, FIG_MODEL)

## CELL 29 — SHAP global feature importance (image + CSV)

In [ ]:
from scipy.sparse import issparse as _iss

print('='*80)
print('SHAP GLOBAL EXPLANATION')
print('='*80)

shap_n     = min(500, X_test_proc.shape[0])
shap_input = X_test_proc[:shap_n]
shap_input = shap_input.toarray() if _iss(shap_input) else shap_input
feat_names = preprocessor.get_feature_names_out().tolist()

print(f'Computing SHAP for {shap_n} test samples ...')
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(shap_input)

if isinstance(shap_values, list):
    mean_abs = np.mean([np.abs(v).mean(0) for v in shap_values], axis=0)
else:
    sv = np.asarray(shap_values)
    mean_abs = (np.abs(sv).mean(axis=(0,2)) if sv.ndim == 3
                else np.abs(sv).mean(0))

shap_df = pd.DataFrame({
    'feature': feat_names, 'mean_abs_shap': mean_abs
}).sort_values('mean_abs_shap', ascending=False)

print('\nTop 30 SHAP features:')
display(shap_df.head(30))
save_csv(shap_df, '22_shap_global_importance.csv')

# Bar chart
plt.figure(figsize=(10, 10))
top20_shap = shap_df.head(20).sort_values('mean_abs_shap')
sns.barplot(data=top20_shap, x='mean_abs_shap', y='feature',
            palette='viridis_r')
plt.title('Top 20 Features — Mean |SHAP| Value (XGBoost)',
          fontsize=13, fontweight='bold')
plt.xlabel('Mean |SHAP value|')
save_fig('11_shap_global_importance.png', L_FIG_SHAP, FIG_SHAP)

# SHAP summary plot
try:
    sv_plot = (shap_values[0] if isinstance(shap_values, list)
               else (np.asarray(shap_values)[:,:,0]
                     if np.asarray(shap_values).ndim==3
                     else np.asarray(shap_values)))
    plt.figure(figsize=(12, 10))
    shap.summary_plot(sv_plot, shap_input,
                      feature_names=feat_names,
                      show=False, max_display=20)
    plt.title('SHAP Summary Plot (XGBoost — Top 20 Features)',
              fontsize=12, fontweight='bold')
    save_fig('12_shap_summary_plot.png', L_FIG_SHAP, FIG_SHAP)
except Exception as e:
    print('SHAP summary plot skipped:', e)

## CELL 30 — Save ALL model artifacts (backend-ready names)

In [ ]:
import pickle as pkl

# Restore stdout before file operations
try:
    sys.stdout.flush()
except Exception: pass
if hasattr(sys.stdout, 'close') and isinstance(sys.stdout, TeeStream):
    sys.stdout.close()
    sys.stdout = _orig_stdout

print('='*80)
print('SAVING ALL MODEL ARTIFACTS')
print('='*80)
print('Required filenames for Flask backend:')
print()

artifacts = {
    'career_model.pkl':           xgb_model,
    'catboost_model.pkl':         cb_model,
    'lgbm_model.pkl':             lgb_model,
    'rf_model.pkl':               rf_model,
    'label_encoder.pkl':          label_encoder,
    'preprocessor.pkl':           preprocessor,
}
for fname, obj in artifacts.items():
    path = L_MODEL / fname
    joblib.dump(obj, path, compress=3)
    print(f'  Saved: {fname}  ({path.stat().st_size/1024:.0f} KB)')

# Pickle files
feat_names_out = preprocessor.get_feature_names_out().tolist()
pkl_artifacts = {
    'ensemble_weights.pkl':      ens_weights,
    'feature_columns.pkl':       feat_names_out,
    'cat_feature_names.pkl':     categorical_features,
    'numeric_feature_names.pkl': numeric_features,
    'cat_feature_indices.pkl':   [],  # OHE — no raw cat indices
}
for fname, obj in pkl_artifacts.items():
    path = L_MODEL / fname
    with open(path, 'wb') as f:
        pkl.dump(obj, f)
    print(f'  Saved: {fname}  ({path.stat().st_size/1024:.1f} KB)')

# Scaler
scaler_obj = preprocessor.named_transformers_['num'].named_steps.get('scaler')
joblib.dump(scaler_obj, L_MODEL / 'scaler.pkl')
print(f'  Saved: scaler.pkl')

# feature_info.json
feature_info = {
    'input_columns':            X.columns.tolist(),
    'numeric_features':         numeric_features,
    'categorical_features':     categorical_features,
    'transformed_feature_names':feat_names_out,
    'target':                   TARGET,
    'remove_specialization':    REMOVE_SPECIALIZATION,
    'career_classes':           label_encoder.classes_.tolist(),
    'num_classes':              len(label_encoder.classes_),
    'test_accuracy':            float(test_accuracy),
    'test_macro_f1':            float(test_macro_f1),
    'top5_accuracy':            float(top5_accuracy),
    'ensemble_weights': {
        'xgboost':      ens_weights[0], 'lightgbm': ens_weights[1],
        'catboost':     ens_weights[2], 'rf':        ens_weights[3],
    },
}
with open(L_MODEL / 'feature_info.json', 'w', encoding='utf-8') as f:
    json.dump(feature_info, f, indent=2)
print(f'  Saved: feature_info.json')

print()
print('All model files:')
for p in sorted(L_MODEL.rglob('*')):
    if p.is_file():
        print(f'  {p.name:40s}  {p.stat().st_size/1024:.0f} KB')

## CELL 31 — Copy ALL outputs to Google Drive

In [ ]:
def copy_tree(src, dst):
    src, dst = Path(src), Path(dst)
    dst.mkdir(parents=True, exist_ok=True)
    n = 0
    for item in src.rglob('*'):
        if item.is_file():
            rel    = item.relative_to(src)
            target = dst / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(item, target)
            n += 1
    return n

print('='*80)
print('COPYING TO GOOGLE DRIVE')
print('='*80)

n1 = copy_tree(L_REPORT_TXT, REPORT_TXT)
n2 = copy_tree(L_REPORT_CSV, REPORT_CSV)
n3 = copy_tree(LOCAL / 'figures', FIG_DIR)
n4 = copy_tree(L_MODEL, MODEL_DIR)

print(f'  Text reports    : {n1} files -> {REPORT_TXT}')
print(f'  CSV  reports    : {n2} files -> {REPORT_CSV}')
print(f'  Figures         : {n3} files -> {FIG_DIR}')
print(f'  Model artifacts : {n4} files -> {MODEL_DIR}')
print()
print('Google Drive folder:', PROJECT_DIR)

# Save run metadata to Drive
metadata = {
    'timestamp':             datetime.now().isoformat(),
    'dataset_path':          str(DATA_PATH),
    'dataset_rows':          int(len(df)),
    'career_classes':        int(len(label_encoder.classes_)),
    'train_rows':            int(len(X_train)),
    'val_rows':              int(len(X_val)),
    'test_rows':             int(len(X_test)),
    'remove_specialization': REMOVE_SPECIALIZATION,
    'use_gpu':               USE_GPU,
    'test_accuracy':         float(test_accuracy),
    'test_macro_f1':         float(test_macro_f1),
    'test_weighted_f1':      float(test_weighted_f1),
    'top3_accuracy':         float(top3_accuracy),
    'top5_accuracy':         float(top5_accuracy),
    'ensemble_weights': {
        'xgboost': ens_weights[0], 'lightgbm': ens_weights[1],
        'catboost': ens_weights[2], 'rf': ens_weights[3],
    }
}
with open(REPORT_DIR / 'run_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print('run_metadata.json saved to Drive.')

## CELL 32 — Final results summary

In [ ]:
print('='*80)
print('FINAL RESULTS SUMMARY')
print('='*80)
summary_rows = [
    ('Dataset rows',      f'{len(df):,}'),
    ('Career classes',    str(len(label_encoder.classes_))),
    ('Train / Val / Test',f'{len(X_train):,} / {len(X_val):,} / {len(X_test):,}'),
    ('Accuracy',          f'{test_accuracy*100:.2f}%'),
    ('Macro Precision',   f'{test_macro_precision*100:.2f}%'),
    ('Macro Recall',      f'{test_macro_recall*100:.2f}%'),
    ('Macro F1',          f'{test_macro_f1*100:.2f}%'),
    ('Weighted F1',       f'{test_weighted_f1*100:.2f}%'),
    ('Top-3 Accuracy',    f'{top3_accuracy*100:.2f}%'),
    ('Top-5 Accuracy',    f'{top5_accuracy*100:.2f}%'),
]
wts = [f'{w:.3f}' for w in ens_weights]
summary_rows.append(('Ensemble weights',
    f'XGB={wts[0]} LGB={wts[1]} CB={wts[2]} RF={wts[3]}'))
summary_rows.append(('Specialization dropped', str(REMOVE_SPECIALIZATION)))

summ_df = pd.DataFrame(summary_rows, columns=['metric','value'])
display(summ_df)
save_csv(summ_df, '23_final_summary.csv')

print('\nDrive folder:', PROJECT_DIR)
print('Models:', MODEL_DIR)
print('\nAll done!')


## Final checklist

| # | Check |
|---|---|
| 1 | `reports/csv/01_missing_values.csv` — no unexpected NaNs |
| 2 | `reports/csv/02_invalid_value_report.csv` — all zeros |
| 3 | `reports/csv/03_career_class_distribution.csv` — note imbalanced classes |
| 4 | `reports/csv/05_outlier_report.csv` — review before removing |
| 5 | Confirm `REMOVE_SPECIALIZATION` matches project scenario |
| 6 | `reports/text/17_classification_report.txt` — per-class F1 |
| 7 | `figures/model/07_confusion_matrix.png` — check off-diagonal |
| 8 | `figures/shap/11_shap_global_importance.png` — top features |
| 9 | Download `models/` folder from Drive into `backend/models/` |
| 10 | Restart Flask server — it auto-loads the new `.pkl` files |

### Backend model file mapping
| File in `models/` | Used by Flask `app.py` |
|---|---|
| `career_model.pkl` | XGBoost |
| `catboost_model.pkl` | CatBoost |
| `lgbm_model.pkl` | LightGBM |
| `rf_model.pkl` | Random Forest |
| `label_encoder.pkl` | Decode career names |
| `preprocessor.pkl` | Transform new inputs |
| `ensemble_weights.pkl` | Soft-vote weights |
| `feature_columns.pkl` | Feature names |
| `cat_feature_names.pkl` | Categorical cols |
| `numeric_feature_names.pkl` | Numeric cols |
| `scaler.pkl` | StandardScaler |
